# Post-Game-Analyse von Fußballspielen mit Machine Learning

**Projektziel:** Erklärung von Fußball-Endergebnissen (`H`, `D`, `A`) anhand von nach dem Spiel verfügbaren Spielstatistiken.

**Wichtige Abgrenzung:** Dieses Notebook ist als **Post-Game-Analyse** aufgebaut. Es soll Zusammenhänge zwischen Spielstatistiken und Endergebnis erklären, nicht vor dem Anpfiff ein Ergebnis prognostizieren.

**Zielvariable:**
- `H` = Heimsieg
- `D` = Unentschieden
- `A` = Auswärtssieg

**Geplante Modelle:**
1. Dummy Classifier als Referenzmodell
2. Logistische Regression als interpretierbare Baseline
3. Random Forest als nicht-lineares Vergleichsmodell

**Hauptmetriken:**
- Macro F1
- Accuracy
- Classification Report
- Confusion Matrix

## 1. Setup und Imports

In diesem Abschnitt werden alle benötigten Bibliotheken geladen.  
Der `RANDOM_STATE` sorgt dafür, dass die Ergebnisse reproduzierbarer sind.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)

## 2. Daten laden

Die CSV-Datei wird geladen.  
Falls das Notebook lokal ausgeführt wird, sollte die Datei `Datensatz_FootballData.csv` im gleichen Ordner wie das Notebook liegen.

In [ ]:
DATA_PATH = Path("Datensatz_FootballData.csv")

if not DATA_PATH.exists():
    DATA_PATH = Path("/mnt/data/Datensatz_FootballData.csv")

df_raw = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df = df_raw.copy()

print(f"Zeilen: {df.shape[0]}")
print(f"Spalten: {df.shape[1]}")
display(df.head())

## 3. Erste Dateninspektion

Ziel dieses Abschnitts ist ein erstes Verständnis des Datensatzes:
- Welche Spalten gibt es?
- Welche Datentypen liegen vor?
- Wie viele fehlende Werte gibt es?
- Wie ist die Zielvariable verteilt?

In [ ]:
display(df.info())
display(df.describe(include="all").T.head(30))

missing_overview = (
    df.isna()
      .mean()
      .mul(100)
      .sort_values(ascending=False)
      .rename("Fehlwerte_in_%")
      .to_frame()
)

display(missing_overview.head(30))

In [ ]:
TARGET = "Endergebnis"

print("Klassenverteilung absolut:")
display(df[TARGET].value_counts(dropna=False))

print("Klassenverteilung relativ:")
display((df[TARGET].value_counts(normalize=True, dropna=False) * 100).round(2))

df[TARGET].value_counts().plot(kind="bar")
plt.title("Klassenverteilung der Zielvariable")
plt.xlabel("Endergebnis")
plt.ylabel("Anzahl Spiele")
plt.show()

## 4. Datumsbereinigung

Im Datensatz können Datumswerte mit zweistelligen und vierstelligen Jahreszahlen vorkommen.  
Deshalb wird eine eigene Funktion verwendet, die beide Formate berücksichtigt.

In [ ]:
def parse_mixed_german_dates(series: pd.Series) -> pd.Series:
    parsed = pd.to_datetime(series, format="%d/%m/%Y", errors="coerce")
    mask = parsed.isna()
    if mask.any():
        parsed.loc[mask] = pd.to_datetime(series.loc[mask], format="%d/%m/%y", errors="coerce")
    return parsed

df["Datum_parsed"] = parse_mixed_german_dates(df["Datum"])

print("Nicht parsebare Datumswerte:", df["Datum_parsed"].isna().sum())
print("Zeitraum:", df["Datum_parsed"].min(), "bis", df["Datum_parsed"].max())

display(df[["Datum", "Datum_parsed", "Heimmannschaft", "Gastmannschaft", TARGET]].head())

## 5. Data-Leakage-Abgrenzung

Da `Endergebnis` direkt aus den finalen Toren ableitbar ist, dürfen finale Ergebnisvariablen nicht als Eingabefeatures verwendet werden.

Für die erste Modellstufe werden deshalb ausgeschlossen:
- `Endergebnis`
- `Endergebnis_Heim_Tore`
- `Endergebnis_Gast_Tore`

Teamnamen werden zunächst ebenfalls nicht als Features verwendet, damit das Modell allgemeine Spielmuster lernt und nicht primär Teamidentitäten.

In [ ]:
leakage_cols = [
    "Endergebnis",
    "Endergebnis_Heim_Tore",
    "Endergebnis_Gast_Tore",
]

meta_cols = [
    "Liga",
    "Datum",
    "Datum_parsed",
    "Uhrzeit",
    "Heimmannschaft",
    "Gastmannschaft",
]

print("Ausgeschlossene Leakage-Spalten:", [c for c in leakage_cols if c in df.columns])
print("Zunächst ausgeschlossene Meta-Spalten:", [c for c in meta_cols if c in df.columns])

## 6. Explorative Datenanalyse der Spielstatistiken

In diesem Abschnitt werden zentrale Spielstatistiken betrachtet.  
Besonders interessant sind Unterschiede zwischen Heim- und Auswärtsteam.

In [ ]:
base_stat_cols = [
    "Heim_Schuesse", "Gast_Schuesse",
    "Heim_Schuesse_aufs_Tor", "Gast_Schuesse_aufs_Tor",
    "Heim_Fouls", "Gast_Fouls",
    "Heim_Ecken", "Gast_Ecken",
    "Heim_Gelbe_Karten", "Gast_Gelbe_Karten",
    "Heim_Rote_Karten", "Gast_Rote_Karten",
    "Halbzeit_Heim_Tore", "Halbzeit_Gast_Tore",
]

existing_stat_cols = [c for c in base_stat_cols if c in df.columns]

display(df.groupby(TARGET)[existing_stat_cols].mean().round(2))

In [ ]:
for col in existing_stat_cols[:8]:
    df.boxplot(column=col, by=TARGET)
    plt.title(f"{col} nach Endergebnis")
    plt.suptitle("")
    plt.xlabel("Endergebnis")
    plt.ylabel(col)
    plt.show()

## 7. Feature Engineering

Aus Rohdaten werden neue, interpretierbare Merkmale gebildet.  
Differenzmerkmale sind im Fußball besonders sinnvoll, weil nicht nur die absolute Anzahl einer Aktion relevant ist, sondern der Vergleich zwischen Heim- und Gastteam.

In [ ]:
def add_diff_features(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()

    diff_pairs = {
        "ShotDiff": ("Heim_Schuesse", "Gast_Schuesse"),
        "ShotOnTargetDiff": ("Heim_Schuesse_aufs_Tor", "Gast_Schuesse_aufs_Tor"),
        "FoulDiff": ("Heim_Fouls", "Gast_Fouls"),
        "CornerDiff": ("Heim_Ecken", "Gast_Ecken"),
        "YellowCardDiff": ("Heim_Gelbe_Karten", "Gast_Gelbe_Karten"),
        "RedCardDiff": ("Heim_Rote_Karten", "Gast_Rote_Karten"),
        "HalfTimeGoalDiff": ("Halbzeit_Heim_Tore", "Halbzeit_Gast_Tore"),
    }

    for new_col, (home_col, away_col) in diff_pairs.items():
        if home_col in data.columns and away_col in data.columns:
            data[new_col] = data[home_col] - data[away_col]

    return data

df_fe = add_diff_features(df)

diff_cols = [
    "ShotDiff",
    "ShotOnTargetDiff",
    "FoulDiff",
    "CornerDiff",
    "YellowCardDiff",
    "RedCardDiff",
    "HalfTimeGoalDiff",
]

existing_diff_cols = [c for c in diff_cols if c in df_fe.columns]
display(df_fe[existing_diff_cols].describe().T)

In [ ]:
display(df_fe.groupby(TARGET)[existing_diff_cols].mean().round(2))

for col in existing_diff_cols:
    df_fe.boxplot(column=col, by=TARGET)
    plt.title(f"{col} nach Endergebnis")
    plt.suptitle("")
    plt.xlabel("Endergebnis")
    plt.ylabel(col)
    plt.show()

## 8. Feature-Auswahl für Modellstufe 1

Für die erste Modellstufe werden nur gut interpretierbare Post-Game-Features verwendet.  
Wettquoten werden zunächst nicht aufgenommen, weil sie eher eine Pre-Game- bzw. Marktinformation darstellen und eine andere Forschungsfrage eröffnen würden.

In [ ]:
numeric_features = [
    "Heim_Schuesse",
    "Gast_Schuesse",
    "Heim_Schuesse_aufs_Tor",
    "Gast_Schuesse_aufs_Tor",
    "Heim_Fouls",
    "Gast_Fouls",
    "Heim_Ecken",
    "Gast_Ecken",
    "Heim_Gelbe_Karten",
    "Gast_Gelbe_Karten",
    "Heim_Rote_Karten",
    "Gast_Rote_Karten",
    "Halbzeit_Heim_Tore",
    "Halbzeit_Gast_Tore",
    "ShotDiff",
    "ShotOnTargetDiff",
    "FoulDiff",
    "CornerDiff",
    "YellowCardDiff",
    "RedCardDiff",
    "HalfTimeGoalDiff",
]

categorical_features = [
    "Halbzeit_Ergebnis",
]

numeric_features = [c for c in numeric_features if c in df_fe.columns]
categorical_features = [c for c in categorical_features if c in df_fe.columns]

feature_cols = numeric_features + categorical_features

print("Numerische Features:")
print(numeric_features)

print("\nKategoriale Features:")
print(categorical_features)

print("\nAnzahl Features vor One-Hot-Encoding:", len(feature_cols))

## 9. Modell-Datensatz erstellen

Zeilen ohne Zielvariable werden ausgeschlossen.  
Fehlende Feature-Werte werden später innerhalb der Pipeline imputiert, damit keine Information aus dem Testset in das Training gelangt.

In [ ]:
model_df = df_fe.dropna(subset=[TARGET]).copy()

X = model_df[feature_cols]
y = model_df[TARGET]

print("X:", X.shape)
print("y:", y.shape)
display(y.value_counts())

## 10. Train-Test-Split

Die Daten werden in Trainings- und Testdaten getrennt.  
Durch `stratify=y` bleibt die Klassenverteilung in beiden Teilmengen ähnlich.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

print("\nKlassenverteilung Training:")
display(y_train.value_counts(normalize=True).round(3))

print("\nKlassenverteilung Test:")
display(y_test.value_counts(normalize=True).round(3))

## 11. Preprocessing-Pipeline

Die Vorverarbeitung wird in einer Pipeline gekapselt:
- numerische Werte: Median-Imputation und Standardisierung
- kategoriale Werte: häufigste Kategorie und One-Hot-Encoding

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)

## 12. Modelle definieren

Es werden drei Modelle verglichen:
1. Dummy Classifier
2. Logistische Regression
3. Random Forest

In [ ]:
models = {
    "Dummy Classifier": Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", DummyClassifier(strategy="most_frequent")),
        ]
    ),
    "Logistische Regression": Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
        ]
    ),
    "Random Forest": Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", RandomForestClassifier(
                n_estimators=400,
                max_depth=None,
                min_samples_leaf=3,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ]
    ),
}

## 13. Training und Evaluation

Die Modelle werden trainiert und anschließend auf dem Testset bewertet.  
Die Hauptmetrik ist `Macro F1`, da sie alle Klassen gleich gewichtet.

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        "Modell": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Macro F1": f1_score(y_test, y_pred, average="macro"),
        "Weighted F1": f1_score(y_test, y_pred, average="weighted"),
    }

fitted_models = {}
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    results.append(evaluate_model(name, model, X_test, y_test))

results_df = pd.DataFrame(results).sort_values("Macro F1", ascending=False)
display(results_df.round(4))

## 14. Classification Reports

Die Reports zeigen je Klasse Precision, Recall und F1.  
Besonders wichtig ist, ob die Klasse `D` für Unentschieden ausreichend erkannt wird.

In [ ]:
for name, model in fitted_models.items():
    print("=" * 80)
    print(name)
    print("=" * 80)
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))

## 15. Confusion Matrices

Die Confusion Matrix zeigt, welche Klassen häufig verwechselt werden.

In [ ]:
class_labels = sorted(y.unique())

for name, model in fitted_models.items():
    fig, ax = plt.subplots(figsize=(5, 5))
    ConfusionMatrixDisplay.from_estimator(
        model,
        X_test,
        y_test,
        labels=class_labels,
        ax=ax,
    )
    ax.set_title(f"Confusion Matrix: {name}")
    plt.show()

## 16. Optionale Cross-Validation

Cross-Validation kann genutzt werden, um die Stabilität der Modellleistung besser einzuschätzen.

In [ ]:
cv_results = []

for name, model in models.items():
    scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring="f1_macro",
        n_jobs=-1,
    )
    cv_results.append({
        "Modell": name,
        "CV Macro F1 Mean": scores.mean(),
        "CV Macro F1 Std": scores.std(),
    })

cv_results_df = pd.DataFrame(cv_results).sort_values("CV Macro F1 Mean", ascending=False)
display(cv_results_df.round(4))

## 17. Interpretation der logistischen Regression

Die Koeffizienten der logistischen Regression können Hinweise darauf geben, welche Merkmale mit welchen Ergebnis-Klassen zusammenhängen.  
Positive Koeffizienten erhöhen die Wahrscheinlichkeit der jeweiligen Klasse im Vergleich zu den anderen Klassen.

In [ ]:
log_model = fitted_models["Logistische Regression"]
feature_names = log_model.named_steps["preprocessor"].get_feature_names_out()
classes = log_model.named_steps["model"].classes_
coef = log_model.named_steps["model"].coef_

coef_df = pd.DataFrame(coef.T, index=feature_names, columns=classes)
coef_df["MaxAbsCoef"] = coef_df.abs().max(axis=1)

display(coef_df.sort_values("MaxAbsCoef", ascending=False).head(25).round(4))

## 18. Feature Importance des Random Forest

Der Random Forest liefert modellinterne Feature Importances.  
Diese sollten vorsichtig interpretiert werden, da korrelierte Merkmale die Wichtigkeiten beeinflussen können.

In [ ]:
rf_model = fitted_models["Random Forest"]
feature_names = rf_model.named_steps["preprocessor"].get_feature_names_out()
importances = rf_model.named_steps["model"].feature_importances_

rf_importance_df = (
    pd.DataFrame({"Feature": feature_names, "Importance": importances})
      .sort_values("Importance", ascending=False)
)

display(rf_importance_df.head(25).round(4))

rf_importance_df.head(15).set_index("Feature")["Importance"].sort_values().plot(kind="barh")
plt.title("Top Feature Importances: Random Forest")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

## 19. Permutation Importance

Permutation Importance misst, wie stark die Modellleistung sinkt, wenn ein Feature zufällig vertauscht wird.  
Hier wird sie auf den Originalfeatures berechnet und ist deshalb für die Interpretation gut geeignet.

In [ ]:
best_model_name = results_df.iloc[0]["Modell"]
best_model = fitted_models[best_model_name]

perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="f1_macro",
    n_repeats=20,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

perm_df = (
    pd.DataFrame({
        "Feature": X_test.columns,
        "Importance_mean": perm.importances_mean,
        "Importance_std": perm.importances_std,
    })
    .sort_values("Importance_mean", ascending=False)
)

print("Bestes Modell nach Test-Macro-F1:", best_model_name)
display(perm_df.round(4))

## 20. Optionaler Zusatz: Wettquoten als eigener Vergleich

Die Wettquoten können in einem separaten Experiment betrachtet werden.  
Das sollte im Bericht aber klar als andere Perspektive formuliert werden, da Quoten eher Markt- bzw. Pre-Game-Informationen darstellen.

Mögliche Forschungsfrage für diesen Zusatz:
> Verbessern Marktinformationen die Klassifikation gegenüber reinen Post-Game-Spielstatistiken?

In [ ]:
odds_keywords = [
    "Markt_Durchschnitt_Heim_Sieg",
    "Markt_Durchschnitt_Unentschieden",
    "Markt_Durchschnitt_Gast_Sieg",
    "Bet365_Heim_Sieg",
    "Bet365_Unentschieden",
    "Bet365_Gast_Sieg",
]

odds_features = [c for c in odds_keywords if c in df_fe.columns]
print("Gefundene Quotenfeatures:", odds_features)

# Dieses Experiment bewusst erst aktivieren, wenn ihr es im Bericht methodisch sauber abgrenzt.
# X_odds = model_df[feature_cols + odds_features]
# Dann analog zu den bisherigen Schritten trainieren und evaluieren.

## 21. Kritische Diskussion

Die Ergebnisse sollten im Portfolio nicht nur technisch, sondern auch fachlich interpretiert werden.

Mögliche Diskussionspunkte:

- **Post-Game statt Pre-Game:** Die Modelle erklären abgeschlossene Spiele und sind kein echtes Prognosemodell vor Anpfiff.
- **Data Leakage:** Finale Tore werden ausgeschlossen, weil sie das Ergebnis direkt bestimmen.
- **Klassenproblem:** Unentschieden sind im Fußball häufig schwerer zu erkennen als Siege.
- **Interpretierbarkeit:** Die logistische Regression ist besser erklärbar; der Random Forest kann dafür nicht-lineare Muster erfassen.
- **Datenqualität:** Fehlende Werte und stark lückenhafte Quotenanbieter-Spalten müssen dokumentiert werden.
- **Erweiterung:** Für eine echte Pre-Game-Prognose müssten ausschließlich vor Spielbeginn bekannte Variablen genutzt werden, z. B. Tabellenposition, Formkurve, Verletzungen, Marktquoten oder historische Teamstärke.

## 22. Fazit-Vorlage

In diesem Projekt wurde ein Machine-Learning-Workflow zur Post-Game-Analyse von Fußballspielen umgesetzt. Die Zielvariable war das dreiklassige Endergebnis (`H`, `D`, `A`). Nach einer explorativen Datenanalyse wurden zentrale Differenzmerkmale gebildet und mehrere Klassifikationsmodelle miteinander verglichen.

Die Ergebnisse zeigen, welches Modell die beste Balance zwischen Genauigkeit und klassenfairer Bewertung erreicht. Besonders relevant ist dabei der Macro-F1-Wert, da er auch die schwächer vertretenen Klassen berücksichtigt. Die Feature-Interpretation liefert Hinweise darauf, welche Spielstatistiken am stärksten mit Heimsieg, Unentschieden oder Auswärtssieg zusammenhängen.

Für eine mögliche Weiterentwicklung wäre eine echte Pre-Game-Prognose interessant. Dafür müsste jedoch ein anderer Feature-Satz verwendet werden, der ausschließlich vor Spielbeginn verfügbare Informationen enthält.